# Les dataclasses — écrire des classes de données sans code répétitif

Une **classe de données**, c'est une classe dont le rôle principal est de **transporter des informations** :
un produit, un utilisateur, un point GPS... Peu de logique, beaucoup d'attributs.

Le problème : en Python classique, écrire une telle classe demande beaucoup de **code répétitif**
(qu'on appelle du *boilerplate* — du code qu'on recopie sans réfléchir) :
- un `__init__` qui recopie chaque paramètre
- un `__repr__` pour l'affichage
- un `__eq__` pour comparer deux objets

Depuis Python 3.7, le décorateur `@dataclass` **génère tout ça automatiquement**.
En 2026, c'est la façon standard d'écrire une classe de données en Python.

---
## 1. Avant / après

### ❌ Avant — tout à la main

In [ ]:
# La version classique : 3 méthodes à écrire pour une simple fiche produit

class ProduitClassique:
    def __init__(self, nom, prix, stock=0):
        self.nom = nom
        self.prix = prix
        self.stock = stock

    def __repr__(self):
        return f"ProduitClassique(nom={self.nom!r}, prix={self.prix!r}, stock={self.stock!r})"

    def __eq__(self, other):
        if not isinstance(other, ProduitClassique):
            return NotImplemented
        return (self.nom, self.prix, self.stock) == (other.nom, other.prix, other.stock)


p = ProduitClassique("Clavier", 49.99)
print(p)
print(p == ProduitClassique("Clavier", 49.99))   # True — même contenu

### ✅ Après — avec `@dataclass`

On déclare seulement les attributs avec leur **type** (les *annotations de type* : `nom: str` veut dire
« nom est censé être une chaîne »). Le décorateur écrit le reste pour nous.

In [ ]:
from dataclasses import dataclass

@dataclass
class Produit:
    nom: str
    prix: float
    stock: int = 0          # valeur par défaut, comme dans un __init__


# __init__, __repr__ et __eq__ sont générés automatiquement :
p = Produit("Clavier", 49.99)
print(p)                                    # __repr__ généré → affichage lisible
print(p == Produit("Clavier", 49.99))       # __eq__ généré → True
print(p == Produit("Souris", 19.99))        # False

---
## 2. Ce que `@dataclass` génère (et ne génère pas)

| Méthode | Générée ? | Rôle |
|---|---|---|
| `__init__` | ✅ oui | reçoit et stocke chaque attribut |
| `__repr__` | ✅ oui | affichage lisible pour le débogage |
| `__eq__` | ✅ oui | deux objets sont égaux si leurs attributs sont égaux |
| `__lt__`, `__le__`... (comparaisons) | ⚙️ avec `order=True` | comparer/trier des objets |
| `__hash__` | ⚙️ avec `frozen=True` | utiliser l'objet dans un `set` ou comme clé de `dict` |

> **Important** : les annotations de type (`nom: str`) sont **indicatives** — Python ne vérifie pas
> les types à l'exécution. `Produit(123, "abc")` ne plante pas. Ce sont les outils comme
> `mypy` ou l'IDE qui s'en servent pour détecter les erreurs avant l'exécution.

---
## 3. Le piège des valeurs par défaut mutables — `field(default_factory=...)`

Rappel du piège classique : une valeur par défaut **mutable** (modifiable, comme une liste)
est créée **une seule fois** et **partagée** entre tous les objets.

`@dataclass` te protège : il **refuse carrément** `tags: list = []` avec une erreur explicite.
La bonne façon : `field(default_factory=list)` — « pour chaque nouvel objet, appelle `list()`
pour fabriquer une liste neuve ».

In [ ]:
from dataclasses import dataclass, field

# ❌ tags: list = []  → ValueError: mutable default ... use default_factory

@dataclass
class Article:
    titre: str
    tags: list[str] = field(default_factory=list)   # une liste NEUVE par objet


a1 = Article("Python et la mémoire")
a2 = Article("Le GIL expliqué")

a1.tags.append("python")
print(f"a1.tags : {a1.tags}")   # ['python']
print(f"a2.tags : {a2.tags}")   # [] — pas de partage, chacun a sa liste
print(f"Même liste ? {a1.tags is a2.tags}")   # False

---
## 4. `frozen=True` — des objets immuables (et donc hashables)

**Immuable** = qu'on ne peut plus modifier après création (comme un tuple ou une string).

Pourquoi c'est utile :
1. **Sécurité** : personne ne peut modifier l'objet par accident.
2. **Hashable** : l'objet peut aller dans un `set` ou servir de clé de `dict`
   (revoir le notebook « sous le capot » : seuls les objets qui ne changent pas peuvent être hashés,
   sinon Python ne retrouverait plus leur « casier »).

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Coordonnee:
    latitude: float
    longitude: float


bruxelles = Coordonnee(50.8503, 4.3517)

# Tenter de modifier → erreur immédiate et claire
try:
    bruxelles.latitude = 0
except Exception as e:
    print(f"{type(e).__name__} : {e}")

# Et comme l'objet ne peut plus changer, il est hashable :
villes_visitees = {bruxelles, Coordonnee(48.8566, 2.3522)}   # dans un set ✔
print(f"\nDans un set : {villes_visitees}")
print(f"Bruxelles déjà visitée ? {Coordonnee(50.8503, 4.3517) in villes_visitees}")   # True, en O(1)

---
## 5. `slots=True` — moins de mémoire, accès plus rapides

Par défaut, chaque objet Python stocke ses attributs dans un dictionnaire interne (`__dict__`).
Pratique (on peut ajouter des attributs à la volée) mais **coûteux en mémoire**.

Avec `slots=True` (Python 3.10+), Python réserve des cases **fixes** pour les attributs déclarés,
sans dictionnaire. Résultat : objets plus légers, accès aux attributs un peu plus rapides.

À utiliser quand tu crées **beaucoup** d'objets (des milliers/millions) — c'est exactement
l'argument « consommation mémoire » qu'on mesure ci-dessous.

In [ ]:
import tracemalloc
from dataclasses import dataclass

@dataclass
class PointClassique:
    x: float
    y: float

@dataclass(slots=True)
class PointSlots:
    x: float
    y: float


N = 100_000

# Mesure mémoire : 100 000 points de chaque type
tracemalloc.start()
points = [PointClassique(i, i) for i in range(N)]
_, pic_classique = tracemalloc.get_traced_memory()
tracemalloc.stop()
del points

tracemalloc.start()
points = [PointSlots(i, i) for i in range(N)]
_, pic_slots = tracemalloc.get_traced_memory()
tracemalloc.stop()
del points

print(f"{N} objets classiques : {pic_classique / 1_000_000:.1f} MB")
print(f"{N} objets slots     : {pic_slots / 1_000_000:.1f} MB")
print(f"→ économie : {(1 - pic_slots / pic_classique) * 100:.0f} %")

# Contrepartie : impossible d'ajouter un attribut non déclaré
p = PointSlots(1, 2)
try:
    p.z = 3
except AttributeError as e:
    print(f"\nAttributeError : {e}")

---
## 6. `__post_init__` — valider ou calculer après la création

Le `__init__` étant généré automatiquement, on ne peut pas y glisser notre logique.
La solution : `__post_init__`, une méthode appelée **juste après** le `__init__` généré.
Parfait pour **valider** les données ou **calculer** un attribut dérivé.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Commande:
    produit: str
    quantite: int
    prix_unitaire: float
    total: float = field(init=False)   # init=False : pas un paramètre du __init__, calculé après

    def __post_init__(self):
        # Validation
        if self.quantite <= 0:
            raise ValueError(f"Quantité invalide : {self.quantite}")
        if self.prix_unitaire < 0:
            raise ValueError(f"Prix invalide : {self.prix_unitaire}")
        # Attribut calculé
        self.total = round(self.quantite * self.prix_unitaire, 2)


cmd = Commande("Clavier", 3, 49.99)
print(cmd)   # le total apparaît dans le __repr__

try:
    Commande("Souris", -1, 19.99)
except ValueError as e:
    print(f"Refusé : {e}")

---
## 7. `order=True` — des objets triables

Avec `order=True`, la dataclass génère les comparaisons (`<`, `<=`, `>`, `>=`) en comparant
les attributs **dans l'ordre de déclaration** — comme un tuple. On peut alors trier une liste d'objets.

In [ ]:
from dataclasses import dataclass

@dataclass(order=True)
class Etudiant:
    moyenne: float      # comparé en premier (ordre de déclaration)
    nom: str            # départage les ex æquo


classe = [
    Etudiant(14.5, "Alice"),
    Etudiant(12.0, "Bob"),
    Etudiant(14.5, "Anna"),
    Etudiant(16.8, "Chloé"),
]

for e in sorted(classe, reverse=True):
    print(f"{e.nom:<8} {e.moyenne}")

---
## 8. Les cousins : `NamedTuple` et `TypedDict`

| Outil | C'est quoi | Quand l'utiliser |
|---|---|---|
| `@dataclass` | une vraie classe, modifiable (sauf `frozen`) | le choix par défaut pour une classe de données |
| `NamedTuple` | un tuple avec des noms de champs | données immuables légères, compatibilité tuple (unpacking, indexation) |
| `TypedDict` | un `dict` avec des clés typées | décrire la forme d'un dictionnaire (ex : un JSON reçu d'une API) |

In [ ]:
from typing import NamedTuple, TypedDict

# NamedTuple : un tuple lisible
class Point(NamedTuple):
    x: float
    y: float

p = Point(3, 4)
print(p)              # Point(x=3, y=4)
print(p.x, p[1])      # accès par nom OU par index (c'est un tuple)
x, y = p              # unpacking comme un tuple

# TypedDict : décrire la forme d'un dict (utile pour les JSON d'API)
class UtilisateurJSON(TypedDict):
    id: int
    nom: str
    actif: bool

u: UtilisateurJSON = {"id": 1, "nom": "Alice", "actif": True}   # un dict normal, mais documenté
print(u["nom"])

---
## Récapitulatif

| Besoin | Solution |
|---|---|
| Classe de données sans boilerplate | `@dataclass` |
| Valeur par défaut liste/dict | `field(default_factory=list)` |
| Objet non modifiable + utilisable dans un set | `@dataclass(frozen=True)` |
| Des millions d'objets → économiser la mémoire | `@dataclass(slots=True)` |
| Valider / calculer à la création | `__post_init__` (+ `field(init=False)`) |
| Trier des objets | `@dataclass(order=True)` |
| Tuple nommé léger | `NamedTuple` |
| Décrire un dict/JSON | `TypedDict` |

> **En entretien** : « Une dataclass génère `__init__`, `__repr__` et `__eq__` à partir des
> annotations de type. Avec `frozen=True` elle devient immuable et hashable, avec `slots=True`
> elle consomme beaucoup moins de mémoire — utile quand on crée des millions d'objets. »